In [1]:
import numpy as np
from PIL import Image
from IPython.display import display

In [2]:
from __future__ import annotations

NUM_ROWS = 28
NUM_COLS = 28
NUM_LABELS = 10

class Data:
  def __init__(self, pixels: list[list[int]], label: int):
    self.pixels = pixels
    self.flattened_pixels: list[int] = []
    for row in self.pixels:
      assert len(row) == NUM_ROWS
      self.flattened_pixels.extend(row)
    
    self.label = label
  
  @classmethod
  def from_csv_row(cls, csv: list[int]) -> Data:
    label = csv[0]
    pixels: list[list[int]] = []
    for row in range(NUM_ROWS):
      start_index = 1 + row * NUM_ROWS
      pixels.append(csv[start_index : start_index + NUM_COLS])

    return cls(pixels, label)
  
  def to_image(self) -> Image:
    pixel_array = np.array(self.pixels, dtype=np.uint8)
    return Image.fromarray(pixel_array, mode='L')
  
  def label_to_one_hot(self) -> list[int]:
    result = [0] * NUM_LABELS
    result[self.label] = 1
    return result

In [3]:
lines: list[str] = []
datas: list[Data] = []

with open('data/MNIST_CSV/mnist_train.csv') as f:
  lines = f.readlines()

for line in lines:
  datas.append(Data.from_csv_row([int(num) for num in line.split(',')]))

In [4]:
print(datas[0].label)
print(datas[0].label_to_one_hot())
display(datas[0].to_image())

5
[0, 0, 0, 0, 0, 1, 0, 0, 0, 0]


In [5]:
distinct_labels = set()
for data in datas:
  distinct_labels.add(data.label)

In [6]:
distinct_labels

{0, 1, 2, 3, 4, 5, 6, 7, 8, 9}

In [7]:
from src.deep_learning.layer import get_random_layer
from src.deep_learning.multi_layer_perceptron import MultiLayerPerceptron
from src.deep_learning.op import TANH
from src.deep_learning.value import Value

multi_layer_perceptron = MultiLayerPerceptron(
    layers=[
        get_random_layer(NUM_ROWS, NUM_ROWS*NUM_COLS, TANH),
        get_random_layer(16, NUM_ROWS, TANH),
        get_random_layer(16, 16, TANH),
        get_random_layer(NUM_LABELS, 16),
    ],
)

In [8]:
data_index = 0

In [9]:
from src.deep_learning.util import softmax
from src.deep_learning.util import cross_entropy

In [11]:
losses: list[float] = []

In [ ]:
for i in range(100):
  # compute loss
  predicted_output = multi_layer_perceptron.forward([float(pixel) for pixel in datas[data_index].flattened_pixels])
  softmax_output = softmax(predicted_output)
  loss = cross_entropy(
      distribution=datas[data_index].label_to_one_hot(),
      predicted_distribution=softmax_output,
  )
  loss.forward()
  losses.append(loss.data)

  # zero gradients
  def zero_gradients(node: Value):
    node.gradient = 0.0
    for child in node.children:
      zero_gradients(child)
  zero_gradients(loss)

  # backwards
  loss.gradient = 1.0
  loss.backward()

  # adjust parameters
  alpha = 0.01
  for layer in multi_layer_perceptron.layers:
    for neuron in layer.neurons:
      for weight_value in neuron.weight_values:
        weight_value.data = weight_value.data + (weight_value.gradient * -alpha)
      neuron.bias_value.data = neuron.bias_value.data + (neuron.bias_value.gradient * -alpha)

  # update state
  data_index = data_index + 1 if data_index < len(datas) - 1 else 0

KeyboardInterrupt: 

In [ ]:
import matplotlib.pyplot as plt

plt.plot(losses)